<a href="https://colab.research.google.com/github/duyanhphan13579dz-dot/Orca-Multi-Finance/blob/main/notebooks/orca_qlora_from_zip_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ORCA - QLoRA train from ZIP 400 samples (Colab T4)

**How to use**
1. Runtime -> Change runtime type -> **T4 GPU**
2. Run Upload cell -> select zip (must contain train.jsonl + val.jsonl)
3. Runtime -> **Run all**
4. Wait for train -> download orca-analyst-lora.zip

LoRA finance: r=32, alpha=64, lr=1.5e-4, 3 epochs.


In [1]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Enable Runtime -> T4 GPU first'
print('GPU:', torch.cuda.get_device_name(0))


Wed Sep 23 00:09:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q -U transformers==4.51.3 peft==0.15.2 trl==0.15.2 bitsandbytes==0.45.4 accelerate datasets sentencepiece protobuf


## 1. Upload ZIP (400 samples)

Run this cell then pick the zip file containing train.jsonl and val.jsonl.


In [2]:
import zipfile
from pathlib import Path
from google.colab import files

print('Select ZIP file to upload...')
uploaded = files.upload()
assert uploaded, 'No file uploaded'

zip_name = list(uploaded.keys())[0]
print('Uploaded:', zip_name)

extract_dir = Path('orca_data')
extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall(extract_dir)
    print('Files:', zf.namelist())

train_path = None
val_path = None
for p in extract_dir.rglob('*.jsonl'):
    name = p.name.lower()
    if name == 'train.jsonl':
        train_path = p
    elif name == 'val.jsonl':
        val_path = p

assert train_path is not None, 'train.jsonl not found in zip'
print('train:', train_path)
print('val:', val_path if val_path else '(none - train only)')

n_train = sum(1 for line in open(train_path, encoding='utf-8') if line.strip())
n_val = sum(1 for line in open(val_path, encoding='utf-8') if line.strip()) if val_path else 0
print('N train:', n_train, '| N val:', n_val)


Select ZIP file to upload...


Saving files (1).zip to files (1) (1).zip
Uploaded: files (1) (1).zip
Files: ['train.jsonl', 'val.jsonl', 'dataset_full_with_category.jsonl', 'HUONG_DAN_TRAINING.md', 'generate_dataset.py']
train: orca_data/train.jsonl
val: orca_data/val.jsonl
N train: 360 | N val: 40


## 2. Load HuggingFace dataset


In [3]:
from datasets import load_dataset

data_files = {'train': str(train_path)}
if val_path is not None:
    data_files['validation'] = str(val_path)

raw = load_dataset('json', data_files=data_files)
print(raw)

sample = raw['train'][0]
print('keys:', sample.keys())
print('roles:', [m['role'] for m in sample['messages']])
print('assistant preview:', sample['messages'][-1]['content'][:200])


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 360
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 40
    })
})
keys: dict_keys(['messages'])
roles: ['system', 'user', 'assistant']
assistant preview: **Tái cân bằng danh mục (Rebalancing)**

- Độ lệch hiện tại: 9 điểm % so với mục tiêu (71% thực tế vs 62% mục tiêu).
- Quy tắc phổ biến: chỉ cần tái cân bằng khi độ lệch vượt ngưỡng 5-10 điểm % (rebal


## 3. Load Qwen2.5-7B 4-bit + LoRA finance (r=32)


In [4]:
!pip uninstall -y bitsandbytes
!pip install bitsandbytes>=0.46.1
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE = 'Qwen/Qwen2.5-7B-Instruct'

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=bnb,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()


Found existing installation: bitsandbytes 0.50.2
Uninstalling bitsandbytes-0.50.2:
  Successfully uninstalled bitsandbytes-0.50.2


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

trainable params: 80,740,352 || all params: 7,696,356,864 || trainable%: 1.0491


## 4. Train (3 epochs, validation if val.jsonl exists)


In [7]:
!pip install trl==0.15.2
from trl import SFTTrainer, SFTConfig

def formatting_func(example):
    return tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )

args = SFTConfig(
    output_dir='./orca-400-out',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,   # tăng để bù batch nhỏ
    learning_rate=1.5e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='no',               # tắt eval khi train để tiết kiệm VRAM
    bf16=True,
    optim='paged_adamw_8bit',
    max_seq_length=1024,              # 2048 → 1024
    packing=False,
    report_to='none',
    gradient_checkpointing=True,      # quan trọng
)

trainer_kw = dict(
    model=model,
    args=args,
    train_dataset=raw['train'],
    processing_class=tokenizer,
    formatting_func=formatting_func,
)
if 'validation' in raw:
    trainer_kw['eval_dataset'] = raw['validation']

trainer = SFTTrainer(**trainer_kw)
trainer.train()
print('TRAIN DONE')


Applying formatting function to train dataset:   0%|          | 0/360 [00:00<?, ? examples/s]

Converting train dataset to ChatML:   0%|          | 0/360 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/360 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/360 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/360 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.98 GiB. GPU 0 has a total capacity of 14.56 GiB of which 393.81 MiB is free. Including non-PyTorch memory, this process has 13.93 GiB memory in use. Of the allocated memory 10.71 GiB is allocated by PyTorch, and 3.09 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## 5. Save adapter + download


In [ ]:
OUT = 'orca-analyst-lora'
model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)

!zip -r orca-analyst-lora.zip orca-analyst-lora
!ls -lh orca-analyst-lora.zip

from google.colab import files
files.download('orca-analyst-lora.zip')
print('Downloaded orca-analyst-lora.zip - next: host with vLLM')


## 6. Deploy vLLM (GPU server, not Colab)

```bash
unzip orca-analyst-lora.zip
vllm serve Qwen/Qwen2.5-7B-Instruct \
  --enable-lora \
  --lora-modules orca-analyst-v1=./orca-analyst-lora \
  --host 0.0.0.0 --port 8000 --max-model-len 4096
```

ORCA `.env`:
```bash
AI_BASE_URL=https://YOUR_HOST/v1
AI_MODEL_ANALYSIS=orca-analyst-v1
AI_MODEL_ANALYSIS_FALLBACKS=qwen/qwen3.8-27b:free,inclusionai/ling-3.0-flash-fin:free
```
